### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_37340/2835454030.py:5: DtypeWarning: Columns (15,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_file_name)


Before preprocessing:
 - n_genes: 39376

After preprocessing:
 - n_genes: 13225


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_37340/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 11841


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    if not set(df_[endpoint_event].dropna().unique()).issubset({0, 1}):
        df_[endpoint_event] = df_[endpoint_event].map({"D": 1, "Yes": 1, "A": 0, "No": 0})
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))

# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)


df_[ordered_cols]

,New OS,vital.status NEW,New RemDur,event,Patient_ID,Cluster_Definition_Post_FAB_Correction,Batches,Platform-RNASeq,Platform-FM1,Platform-RPPA,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,84.00,1,21.14,1,3003486-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,2.226803,1.146102,2.057132,1.723706,1.219951,1.308459,2.791821,2.582067,1.570612,1.833584
1,8.29,1,NaN,0,3044356-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,4.098751,2.785802,4.125153,4.457178,2.766367,2.800134,4.732233,4.073847,2.629900,4.112465
2,22.43,1,NaN,0,3091926-N,Cluster_1,Batch1,RNASeq,FM1,RPPA,...,4.953971,3.637522,4.966192,5.198860,3.386643,3.627556,5.615686,4.996264,3.570893,5.019097
3,129.29,1,111.71,1,3103988-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,3.943428,2.473518,3.655212,3.787883,2.247079,2.565096,4.512628,4.068320,2.731174,3.711038
4,36.57,1,NaN,0,3118298-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,3.415620,1.849062,3.000872,2.935574,1.845778,1.992381,3.951988,3.377248,1.986683,3.110679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,9.57,1,NaN,0,3306954-N,Cluster_1,Batch3A,RNASeq,FM1,RPPA,...,3.133795,1.834739,3.014047,3.162303,1.678322,1.956323,3.908274,3.494114,2.151255,3.158870
77,19.71,1,NaN,0,3308122-N,Cluster_3,Batch3A,RNASeq,FM1,RPPA,...,3.812149,2.177566,3.386528,3.455427,2.183473,2.608776,4.497684,4.282676,2.924176,3.838753
78,33.71,1,13.29,1,3319790-N,Cluster_3,Batch3A,RNASeq,FM1,RPPA,...,3.418587,1.802092,2.974382,2.977984,1.953203,2.017852,4.004153,3.803146,2.323010,3.194763
79,1.29,1,NaN,0,3282000-N,Cluster_2,Batch3A,RNASeq,FM1,RPPA,...,2.522748,1.157187,2.279789,2.544884,1.305132,1.056627,2.925882,2.593444,1.365178,2.428949
